In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType,LongType,DoubleType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("srcData2", "adlsrcset2")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
srcData2 = dbutils.widgets.get("srcData2")

ruta = f"abfss://{container}@{srcData2}.dfs.core.windows.net/transacciones.json"

In [0]:
transacciones_schema = StructType(fields=[
    StructField("transaccion_id", StringType(), False),
    StructField("cliente_id", LongType(), True),
    StructField("monto_operacion", DoubleType(), True),
    StructField("frecuencia_transacciones", IntegerType(), True),
    StructField("uso_cajeros", IntegerType(), True),
    StructField("uso_agencias", IntegerType(), True),
    StructField("uso_pos", IntegerType(), True),
    StructField("cambios_clave_recientes", IntegerType(), True),
    StructField("historial_suspicious", IntegerType(), True),
    StructField("ubicacion_inusual", IntegerType(), True),
    StructField("horario_inusual", IntegerType(), True),
    StructField("dispositivo_nuevo", IntegerType(), True),
    StructField("fallas_autenticacion", IntegerType(), True),
    StructField("intentos_contacto_sospechoso", IntegerType(), True),
    StructField("operaciones_canceladas", IntegerType(), True),
    StructField("alerta_sistema", IntegerType(), True),
    StructField("fraude_confirmado", IntegerType(), True)
])

In [0]:
df_transacciones = spark.read \
    .schema(transacciones_schema) \
    .option("multiLine", False) \
    .json(ruta)

In [0]:
transacciones_final_df = df_transacciones.select(
    col("transaccion_id"),
    col("cliente_id"),
    col("monto_operacion"),
    col("frecuencia_transacciones"),
    col("uso_cajeros"),
    col("uso_agencias"),
    col("uso_pos"),
    col("cambios_clave_recientes"),
    col("historial_suspicious"),
    col("ubicacion_inusual"),
    col("horario_inusual"),
    col("dispositivo_nuevo"),
    col("fallas_autenticacion"),
    col("intentos_contacto_sospechoso"),
    col("operaciones_canceladas"),
    col("alerta_sistema"),
    col("fraude_confirmado")
).withColumn("_fecha_ingesta", current_timestamp())

In [0]:
transacciones_final_df.write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.transacciones")

In [0]:
# 6. Verificación de Ingesta
print("Total de transacciones cargadas en Bronze:", spark.table(f"{catalogo}.{esquema}.transacciones").count())